Overwrite silver files by:
1) Changing state abbreviations in members_119.json
2) Resolving issue with incorrect id for senate rollcall data by:
    - Mapping member_id to "{member last name} ({party}-{state})" to get correct bioguide for each senator
    - Rewrite silver file to replace rollcall_id with member_id

In [ ]:
# CHANGING STATE TO STATE ABBREVIATIONS

import csv
import json

def load_state_mapping(csv_path):
    mapping = {}

    with open(csv_path, newline="", encoding="utf-8") as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            full_name = row["State"].strip()
            abbr = row["Abbreviation"].strip()
            mapping[full_name] = abbr
    return mapping

# Replaces all 'state' values in a JSON file w/ abbreviation
def replace_state_values(json_path, state_mapping):
    """
    Replaces all 'state' values in a JSON file
    with the corresponding abbreviation.
    """

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Traverse structure: pages -> members -> state
    for page in data.get("pages", []):
        for member in page.get("members", []):
            full_state = member.get("state")
            if full_state in state_mapping:
                member["state"] = state_mapping[full_state]

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


csv_path = "../data/references/states.csv"
json_path = "../data/silver/members_119.json"

state_mapping = load_state_mapping(csv_path)
replace_state_values(json_path, state_mapping)


In [5]:
# Builds a map: "LastName (P-ST)" -> bioguide_id
def build_member_lookup(members_path):
    with open(members_path, "r", encoding="utf-8") as f:
        members = json.load(f)

    lookup = {}

    for m in members:
        last_name = m["last_name"].strip()
        party = m["party"].strip()
        state = m["state_name"].strip()
        bioguide_id = m["member_id"].strip()

        key = f"{last_name} ({party}-{state})"
        lookup[key] = bioguide_id

    return lookup

# Adds bioguide_id field to each vote entry.
def attach_bioguide_ids(votes_path, member_lookup):


    with open(votes_path, "r", encoding="utf-8") as f:
        vote_data = json.load(f)

    for vote in vote_data.get("votes", []):
        member_key = vote.get("member")

        if member_key in member_lookup:
            vote["member_id"] = member_lookup[member_key]
        else:
            vote["member_id"] = None  # unmatched case

    with open(votes_path, "w", encoding="utf-8") as f:
        json.dump(vote_data, f, indent=2)


members_file = "../data/gold/members_119.json"
votes_file_1 = "../data/silver/senate_votes_119_1.json"
votes_file_2 = "../data/silver/senate_votes_119_2.json"

member_lookup = build_member_lookup(members_file)

attach_bioguide_ids(votes_file_1, member_lookup)
attach_bioguide_ids(votes_file_2, member_lookup)